# Gromacs simulation setup for basic protein MD

Heavily based on the famous [Lysosome tutorial](http://www.mdtutorials.com/gmx/lysozyme/).

The dashboard provides you with the molecule as `input.pdb`
First, give it a nice name, specify the main simulation length, and adjust number of equilibration steps if necessary

In [ ]:
# Simulation parameters
name = "hammerhead"  # give me a better name

nanoseconds = 10.  # just for fun
nsteps = int(nanoseconds * 1000 * 500)  # assumes usual 2fs integration step

# FIXME
eqsteps = 20000  # works for small proteins but can be a bit unrealistic (20ps)

# Engine CLI overrides
extra_args = ""

# Output directories
temp_dir = "temp"
analysis_dir = "analysis"
production_dir = "production"

## Just import what we need

In [ ]:
import json
from pathlib import Path

import gromacs as gmx
import matplotlib.pyplot as plt
import mdtraj as md
import nglview as nv
import numpy as np
import glob

for d in (temp_dir, analysis_dir, production_dir):
    Path(d).mkdir(parents=True, exist_ok=True)
    (Path(d) / ".gitkeep").touch()

## Look at the input

Tune NGLView parameters if needed, inspect the input visually

In [ ]:
nv.show_file("input.pdb")

## Initial setup

### Parse the input

Let Gromacs parse the input file, specify water model and force field. Check and fix eventual errors.



In [ ]:
gmx.pdb2gmx(f="input.pdb", o=f"{temp_dir}/{name}.gro", water="tip3p", ff="amber99sb-ildn", p=f"{temp_dir}/{name}.top", ignh=True)
for itp in glob.glob("posre_*.itp"):
    Path(itp).rename(f"{temp_dir}/{itp}")

### Simulation box

Set up the box, choose it's shape and adjust size eventually.

In [ ]:
gmx.editconf(f=f"{temp_dir}/{name}.gro", o=f"{temp_dir}/{name}-box.gro", c=True, d="2.5", bt="dodecahedron")

### Solvate
Add water to the molecule.

Gromacs overwrites the topology file in this stage, therefore we rename it first.

In [ ]:
!cp {temp_dir}/{name}.top {temp_dir}/{name}-solv.top
gmx.solvate(cp=f"{temp_dir}/{name}-box.gro", cs="spc216.gro", o=f"{temp_dir}/{name}-solv.gro", p=f"{temp_dir}/{name}-solv.top")

### Add counterions

Add Cl- or Na+ to compensate for charge. Adjust parameters here if physiological salt concetntration is required etc.

In [ ]:
with open(f"{temp_dir}/ions.mdp", "w") as ions:
    ions.write("""\
integrator  = steep         ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0        ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.01          ; Minimization step size
nsteps      = 50000         ; Maximum number of (minimization) steps to perform

; Parameters describing how to find the neighbors of each atom and how to calculate the interactions
nstlist         = 1         ; Frequency to update the neighbor list and long range forces
cutoff-scheme	= Verlet    ; Buffered neighbor searching 
ns_type         = grid      ; Method to determine neighbor list (simple, grid)
coulombtype     = cutoff    ; Treatment of long range electrostatic interactions
rcoulomb        = 1.0       ; Short-range electrostatic cut-off
rvdw            = 1.0       ; Short-range Van der Waals cut-off
pbc             = xyz       ; Periodic Boundary Conditions in all 3 dimensions
""")

gmx.grompp(f=f"{temp_dir}/ions.mdp", c=f"{temp_dir}/{name}-solv.gro", p=f"{temp_dir}/{name}-solv.top", o=f"{temp_dir}/ions.tpr", po=f"{temp_dir}/ions_mdout.mdp")

In [ ]:
gmx.select(s=f"{temp_dir}/{name}-solv.gro", on=f"{temp_dir}/solv.ndx", select="SOL")

In [ ]:
!cp {temp_dir}/{name}-solv.top {temp_dir}/{name}-ions.top
gmx.genion(s=f"{temp_dir}/ions.tpr", n=f"{temp_dir}/solv.ndx", o=f"{temp_dir}/{name}-ions.gro", p=f"{temp_dir}/{name}-ions.top", pname="NA", nname="CL", neutral=True)

### Minimize

Gradient descend the energy of the system we set up so that simulation does note explode.

Visualize the minimization trajectory to check something does not go wrong.

In [ ]:
with open(f"{temp_dir}/minim.mdp", "w") as m:
    m.write("""\
integrator  = steep         ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0        ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.005          ; Minimization step size
nsteps      = 50000         ; Maximum number of (minimization) steps to perform

; Parameters describing how to find the neighbors of each atom and how to calculate the interactions
nstlist         = 1         ; Frequency to update the neighbor list and long range forces
cutoff-scheme   = Verlet    ; Buffered neighbor searching
ns_type         = grid      ; Method to determine neighbor list (simple, grid)
coulombtype     = PME       ; Treatment of long range electrostatic interactions
rcoulomb        = 1.0       ; Short-range electrostatic cut-off
rvdw            = 1.0       ; Short-range Van der Waals cut-off
pbc             = xyz       ; Periodic Boundary Conditions in all 3 dimensions

nstxout                 = 50         
nstvout                 = 0        
nstfout                 = 0
nstenergy               = 50         
""")

gmx.grompp(f=f"{temp_dir}/minim.mdp", c=f"{temp_dir}/{name}-ions.gro", p=f"{temp_dir}/{name}-ions.top", o=f"{temp_dir}/em.tpr", po=f"{temp_dir}/em_mdout.mdp")

In [ ]:
gmx.mdrun(deffnm=f"{temp_dir}/em")

In [ ]:
gmx.select(s=f"{temp_dir}/{name}-ions.gro", on=f"{temp_dir}/dnarna.ndx", select="resname A C G U DA DC DG DT")

In [ ]:
gmx.trjconv(s=f"{temp_dir}/{name}-ions.gro", f=f"{temp_dir}/em.trr", n=f"{temp_dir}/dnarna.ndx", o=f"{temp_dir}/em-dnarna.xtc")

In [ ]:
gmx.trjconv(s=f"{temp_dir}/{name}-ions.gro", f=f"{temp_dir}/{name}.gro", n=f"{temp_dir}/dnarna.ndx", o=f"{analysis_dir}/{name}-reference.gro")
tr = md.load(f"{temp_dir}/em-dnarna.xtc", top=f"{analysis_dir}/{name}-reference.gro")
nv.show_mdtraj(tr)

## Equilibration

### Constant Number-Volume-Temperature

Assing some initial velocities to the atoms, restrain protein heavy atom positions, apply thermostat and run short simulation.

Observe the resulting pressure and temperature curves. They can oscillate but should not show any trend apart of an initial phase.

In [ ]:
gmx.make_ndx(f=f"{temp_dir}/{name}-ions.gro",o=f"{temp_dir}/default.ndx",input="q")
!cat {temp_dir}/default.ndx {temp_dir}/dnarna.ndx >{temp_dir}/all.ndx

In [ ]:
with open(f"{temp_dir}/nvt.mdp", "w") as nvt:
    nvt.write(f"""title                   = NVT equilibration 
define                  = -DPOSRES  ; position restrain the protein
; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = {eqsteps}     ; 2 * 50000 = 100 ps
dt                      = 0.002     ; 2 fs
; Output control
nstxout                 = 500       ; save coordinates every 1.0 ps
nstvout                 = 500       ; save velocities every 1.0 ps
nstenergy               = 500       ; save energies every 1.0 ps
nstlog                  = 500       ; update log file every 1.0 ps
; Bond parameters
continuation            = no        ; first dynamics run
constraint_algorithm    = lincs     ; holonomic constraints 
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy
; Nonbonded settings 
cutoff-scheme           = Verlet    ; Buffered neighbor searching
ns_type                 = grid      ; search neighboring grid cells
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT
; Temperature coupling is on
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = resname_A_C_G_U_DA_DC_DG_DT Water_and_ions   ; two coupling groups - more accurate
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300     300           ; reference temperature, one for each group, in K
; Pressure coupling is off
pcoupl                  = no        ; no pressure coupling in NVT
; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC
; Velocity generation
gen_vel                 = yes       ; assign velocities from Maxwell distribution
gen_temp                = 300       ; temperature for Maxwell distribution
gen_seed                = -1        ; generate a random seed
""")
gmx.grompp(f=f"{temp_dir}/nvt.mdp", c=f"{temp_dir}/em.gro", r=f"{temp_dir}/em.gro", p=f"{temp_dir}/{name}-ions.top", n=f"{temp_dir}/all.ndx", o=f"{temp_dir}/nvt.tpr", po=f"{temp_dir}/nvt_mdout.mdp")

In [ ]:
gmx.mdrun(deffnm=f"{temp_dir}/nvt", pin="on")

In [ ]:
gmx.energy(f=f"{temp_dir}/nvt.edr", o=f"{temp_dir}/press.xvg", input="Pressure")
gmx.energy(f=f"{temp_dir}/nvt.edr", o=f"{temp_dir}/temp.xvg", input="Temperature")

In [ ]:
temp = np.loadtxt(f"{temp_dir}/temp.xvg", comments=["#", "@"])
press = np.loadtxt(f"{temp_dir}/press.xvg", comments=["#", "@"])

plt.figure(figsize=(15, 5))
plt.subplot(211)
plt.plot(press[:, 0], press[:, 1])
plt.title("isothermal-isochoric equilibration")
plt.grid()
# plt.xlabel('time (ps)')
plt.ylabel("pressure (bar)")


plt.subplot(212)
plt.xlabel("time (ps)")
plt.ylabel("temperature (K)")
plt.grid()
plt.plot(temp[:, 0], temp[:, 1])

plt.show()

### Constant Number-Pressure-Temperature

Keep the heavy atom positions restrained, besides thermostat apply also pressure coupling.

Observer pressure, temperature, and density, which should ramp up again and remain oscillating around reasonable values.

In [ ]:
with open(f"{temp_dir}/npt.mdp", "w") as npt:
    npt.write(f"""define                  = -DPOSRES  ; position restrain the protein
; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = {eqsteps}     ; 2 * 50000 = 100 ps
dt                      = 0.002     ; 2 fs
; Output control
nstxout                 = 500       ; save coordinates every 1.0 ps
nstvout                 = 500       ; save velocities every 1.0 ps
nstenergy               = 500       ; save energies every 1.0 ps
nstlog                  = 500       ; update log file every 1.0 ps
; Bond parameters
continuation            = yes       ; Restarting after NVT 
constraint_algorithm    = lincs     ; holonomic constraints 
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy
; Nonbonded settings 
cutoff-scheme           = Verlet    ; Buffered neighbor searching
ns_type                 = grid      ; search neighboring grid cells
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet scheme
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT
; Temperature coupling is on
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = resname_A_C_G_U_DA_DC_DG_DT Water_and_ions   ; two coupling groups - more accurate
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300     300           ; reference temperature, one for each group, in K
; Pressure coupling is on
; ljocha pcoupl                  = Parrinello-Rahman     ; Pressure coupling on in NPT
pcoupl = C-rescale
pcoupltype              = isotropic             ; uniform scaling of box vectors
; ljocha tau_p                   = 2.0                   ; time constant, in ps
tau_p = 5.0
ref_p                   = 1.0                   ; reference pressure, in bar
compressibility         = 4.5e-5                ; isothermal compressibility of water, bar^-1
refcoord_scaling        = com
; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC
; Velocity generation
gen_vel                 = no        ; Velocity generation is off 
""")
gmx.grompp(f=f"{temp_dir}/npt.mdp", c=f"{temp_dir}/nvt.gro", r=f"{temp_dir}/nvt.gro", p=f"{temp_dir}/{name}-ions.top", n=f"{temp_dir}/all.ndx", o=f"{temp_dir}/npt.tpr", po=f"{temp_dir}/npt_mdout.mdp")

In [ ]:
gmx.mdrun(deffnm=f"{temp_dir}/npt", pin="on")

In [ ]:
gmx.energy(f=f"{temp_dir}/npt.edr", o=f"{temp_dir}/press.xvg", input="Pressure")
gmx.energy(f=f"{temp_dir}/npt.edr", o=f"{temp_dir}/dens.xvg", input="Density")
gmx.energy(f=f"{temp_dir}/npt.edr", o=f"{temp_dir}/temp.xvg", input="Temperature")

In [ ]:
temp = np.loadtxt(f"{temp_dir}/temp.xvg", comments=["#", "@"])
press = np.loadtxt(f"{temp_dir}/press.xvg", comments=["#", "@"])
dens = np.loadtxt(f"{temp_dir}/dens.xvg", comments=["#", "@"])

plt.figure(figsize=(15, 7))
plt.subplot(311)
plt.plot(press[:, 0], press[:, 1])
plt.title("isothermal-isobaric equilibration")
plt.grid()
# plt.xlabel('time (ps)')
plt.ylabel("pressure (bar)")

plt.subplot(312)
plt.ylabel("density (kg/m3)")
plt.grid()
plt.plot(dens[:, 0], dens[:, 1])

plt.subplot(313)
plt.xlabel("time (ps)")
plt.ylabel("temperature (K)")
plt.grid()
plt.plot(temp[:, 0], temp[:, 1])

plt.show()

## Production Handoff

Generate the production run input, then proceed to the next step (production) in the Dashboard.

In [ ]:
with open(f"{production_dir}/{name}.mdp", "w") as mdp:
    mdp.write(f"""integrator              = md        ; leap-frog integrator
dt                      = 0.002     ; 2 fs
; Output control
nstxout                 = 0         ; suppress bulky .trr file by specifying 
nstvout                 = 0         ; 0 for output frequency of nstxout,
nstfout                 = 0         ; nstvout, and nstfout
nstenergy               = 5000      ; save energies every 10.0 ps
nstlog                  = 5000      ; update log file every 10.0 ps
nstxout-compressed      = 5000      ; save compressed coordinates every 10.0 ps
compressed-x-grps       = resname_A_C_G_U_DA_DC_DG_DT    
; Bond parameters
continuation            = yes       ; Restarting after NPT 
constraint_algorithm    = lincs     ; holonomic constraints 
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy
; Neighborsearching
cutoff-scheme           = Verlet    ; Buffered neighbor searching
ns_type                 = grid      ; search neighboring grid cells
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet scheme
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT
; Temperature coupling is on
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = resname_A_C_G_U_DA_DC_DG_DT Water_and_ions   ; two coupling groups - more accurate
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300 300           ; reference temperature, one for each group, in K
; Pressure coupling is on
pcoupl                  = Parrinello-Rahman     ; Pressure coupling on in NPT
pcoupltype              = isotropic             ; uniform scaling of box vectors
tau_p                   = 2.0                   ; time constant, in ps
ref_p                   = 1.0                   ; reference pressure, in bar
compressibility         = 4.5e-5                ; isothermal compressibility of water, bar^-1
; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC
; Dispersion correction
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Velocity generation
gen_vel                 = no        ; Velocity generation is off 
nsteps = {nsteps}
""")

gmx.grompp(f=f"{production_dir}/{name}.mdp", c=f"{temp_dir}/npt.gro", r=f"{temp_dir}/npt.gro", p=f"{temp_dir}/{name}-ions.top", n=f"{temp_dir}/all.ndx", o=f"{production_dir}/{name}.tpr", po=f"{temp_dir}/{name}_mdout.mdp")

In [ ]:
sim = {
    "$schema": "https://raw.githubusercontent.com/CERIT-SC/mddash/v0.1.4/dashboard/api/manifest_schemas/gromacs.schema.json",
    "name": name,
    "engine": "GMX",
    "files": {
        "run_input": f"{production_dir}/{name}.tpr",
        "run_structure": f"{production_dir}/{name}.gro",
        "reference_structure": f"{analysis_dir}/{name}-reference.gro",
        "trajectory": f"{production_dir}/{name}.xtc",
    },
    "extra_args": extra_args,
}
sim_file = Path(f"{name}.simulation.json")
sim_file.write_text(json.dumps(sim, indent=2, sort_keys=True))
print(f"Wrote {sim_file}")